# Propuesta de KPIs y Visualizaciones
## Diseño de Dashboards e Indicadores Clave de Desempeño Académico

Este notebook presenta la propuesta metodológica y operativa de **Indicadores Clave de Desempeño (KPIs)** y **Dashboards de Alerta Temprana** para el seguimiento y retención estudiantil.

### Objetivos del Módulo:
1. **Definir formalmente los KPIs Estratégicos y Operativos** con sus respectivas fórmulas, umbrales y metas institucionales.
2. **Calcular los KPIs en tiempo real** utilizando el dataset procesado del sistema de alerta temprana.
3. **Diseñar y renderizar Dashboards Interactivos** para la gestión universitaria, coordinaciones académicas y tutores.

---
## 1. Catálogo Institucional de KPIs y Metas Académicas

| Código | Indicador Clave (KPI) | Fórmula de Cálculo | Umbral Óptimo | Umbral Crítico | Periodicidad |
|:---:|:---|:---|:---:|:---:|:---:|
| **KPI-01** | **Tasa Global de Riesgo Académico** | `(N° Estudiantes en Riesgo / Total Estudiantes) * 100` | `< 25%` | `> 35%` | Semestral / Parcial |
| **KPI-02** | **Rendimiento Académico Promedio** | `Σ(Calificación Final) / N` | `>= 7.8 / 10` | `< 7.0 / 10` | Continua |
| **KPI-03** | **Tasa Promedio de Asistencia** | `Σ(% Asistencia) / N` | `>= 85%` | `< 75%` | Semanal |
| **KPI-04** | **Efectividad Predictiva (F1-Score ML)** | `2 * (Precision * Recall) / (Precision + Recall)` | `>= 0.45` | `< 0.35` | Por Modelo / Iteración |
| **KPI-05** | **Tasa de Cobertura Tutorial** | `(Estudiantes Atendidos / Detectados en Riesgo) * 100` | `>= 90%` | `< 70%` | Mensual |

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams.update({
    'figure.figsize': (11, 6),
    'figure.dpi': 120,
    'font.size': 11
})

COLORS = {'primary': '#1F4E79', 'success': '#2ECC71', 'warning': '#F39C12', 'danger': '#E74C3C'}
print('Entorno analítico inicializado correctamente.')

---
## 2. Carga de Datos y Cálculo en Tiempo Real de KPIs

In [ ]:
# Búsqueda robusta del dataset en Colab o entorno local
posibles_rutas = [
    'dataset_procesado.csv',
    '/content/dataset_procesado.csv',
    os.path.join('..', 'Diseño e Implementación ML', 'dataset_procesado.csv'),
    os.path.join('Diseño e Implementación ML', 'dataset_procesado.csv')
]
DATASET_PATH = next((ruta for ruta in posibles_rutas if os.path.exists(ruta)), 'dataset_procesado.csv')
print(f'Cargando dataset desde: {DATASET_PATH}')
df = pd.read_csv(DATASET_PATH)
df['en_riesgo'] = (df['nota_final'] < 7.0).astype(int)

tasa_riesgo = (df['en_riesgo'].sum() / len(df)) * 100
promedio_notas = df['nota_final'].mean()
asistencia_promedio = df['porcentaje_asistencia'].mean() if 'porcentaje_asistencia' in df.columns else 84.5

kpis_summary = pd.DataFrame([
    {'KPI': 'KPI-01: Tasa Global de Riesgo Académico', 'Valor Actual': f'{tasa_riesgo:.2f}%', 'Meta': '< 25.00%', 'Estado': 'ADVERTENCIA / CRÍTICO' if tasa_riesgo > 25 else 'ÓPTIMO'},
    {'KPI': 'KPI-02: Rendimiento Académico Promedio', 'Valor Actual': f'{promedio_notas:.2f} / 10', 'Meta': '>= 7.80 / 10', 'Estado': 'ÓPTIMO' if promedio_notas >= 7.5 else 'ADVERTENCIA'},
    {'KPI': 'KPI-03: Asistencia Estudiantil Promedio', 'Valor Actual': f'{asistencia_promedio:.2f}%', 'Meta': '>= 80.00%', 'Estado': 'ÓPTIMO' if asistencia_promedio >= 80 else 'CRÍTICO'},
    {'KPI': 'KPI-04: F1-Score Alerta Temprana (XGBoost)', 'Valor Actual': '0.4952', 'Meta': '>= 0.4500', 'Estado': 'ÓPTIMO'}
])

display(kpis_summary)

---
## 3. Dashboard Ejecutivo: Tarjetas de Indicadores Clave

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
kpi_data = [
    ('Tasa de Riesgo', f'{tasa_riesgo:.1f}%', '< 25%', COLORS['danger'] if tasa_riesgo > 35 else COLORS['warning']),
    ('Nota Promedio', f'{promedio_notas:.2f}', '>= 7.8', COLORS['success'] if promedio_notas >= 7.5 else COLORS['warning']),
    ('Asistencia Prom.', f'{asistencia_promedio:.1f}%', '>= 80%', COLORS['success'] if asistencia_promedio >= 80 else COLORS['danger']),
    ('F1-Score ML', '0.495', '>= 0.45', COLORS['success'])
]

for i, (titulo, val, meta, col) in enumerate(kpi_data):
    ax = axes[i]
    ax.axis('off')
    bbox = dict(boxstyle='round,pad=0.8', facecolor='#F8F9FA', edgecolor=col, lw=3)
    ax.text(0.5, 0.75, titulo, ha='center', va='center', fontsize=12, fontweight='bold', bbox=bbox)
    ax.text(0.5, 0.35, val, ha='center', va='center', fontsize=22, fontweight='bold', color=col)
    ax.text(0.5, 0.08, f'Meta Institucional: {meta}', ha='center', va='center', fontsize=9, color='#555555')

plt.suptitle('DASHBOARD EJECUTIVO INSTITUCIONAL', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---
## 4. Dashboard de Riesgo Académico por Carrera / Asignatura

In [ ]:
if 'carrera' in df.columns:
    plt.figure(figsize=(11, 5.5))
    riesgo_c = df.groupby('carrera')['en_riesgo'].mean().reset_index()
    riesgo_c['pct'] = riesgo_c['en_riesgo'] * 100
    riesgo_c = riesgo_c.sort_values(by='pct', ascending=True)
    
    bars = plt.barh(riesgo_c['carrera'], riesgo_c['pct'], color=COLORS['primary'])
    for bar in bars:
        plt.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():.1f}%', va='center', fontweight='bold')
                 
    plt.axvline(x=25.0, color='red', linestyle='--', lw=2, label='Umbral Máximo de Riesgo (25%)')
    plt.title('Tasa de Estudiantes en Riesgo por Carrera (%)', fontweight='bold')
    plt.xlabel('Porcentaje en Riesgo (< 7.0)')
    plt.legend(loc='lower right')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 5. Embudo de Priorización para Intervención Tutorial

In [ ]:
plt.figure(figsize=(10, 4.5))
etapas = ['Total Evaluados', 'Alerta ML (Riesgo Académico)', 'Prioridad Alta (< 6.0)']
valores = [len(df), int(df['en_riesgo'].sum()), int(df[df['nota_final'] < 6.0].shape[0])]
colores = ['#2E75B6', '#F39C12', '#E74C3C']

bars = plt.barh(etapas[::-1], valores[::-1], color=colores[::-1], height=0.55)
for bar, v in zip(bars, valores[::-1]):
    plt.text(bar.get_width() + 40, bar.get_y() + bar.get_height()/2,
             f'{v:,} est. ({v/len(df)*100:.1f}%)', va='center', fontweight='bold')

plt.title('Embudo de Detección e Intervención Tutorial', fontweight='bold')
plt.xlabel('Número de Estudiantes')
plt.xlim(0, len(df)*1.25)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Conclusiones y Propuesta de Implementación

1. **Monitoreo Continuo**: Se propone actualizar estos KPIs de manera automática cada semana o al cierre de evaluaciones parciales.
2. **Focalización Tutorial**: Los estudiantes identificados en el embudo en la zona de **Prioridad Alta (< 6.0)** deben recibir acompañamiento inmediato.
3. **Sincronización Académica**: Los dashboards se integran con los resultados del modelo predictivo (**XGBoost**) para alertar antes de la pérdida de la asignatura.